# Bab 6: Membangun Jaringan Saraf Mendalam Pertama Anda: Pengantar Backpropagation
*(Building Your First Deep Neural Network: Intro to Backpropagation)*

## Eksplorasi Teoritis Mendalam (Theoretical Deep-Dive)

Bab ini adalah salah satu tonggak paling krusial dalam memahami Deep Learning. Pada bab sebelumnya, jaringan kita hanya memiliki dua lapisan: Input dan Output. Model tersebut sangat terbatas karena hanya bisa mempelajari hubungan linier sederhana. Untuk memecahkan masalah dunia nyata yang rumit (seperti pengenalan gambar atau terjemahan bahasa), kita harus menumpuk matriks bobot menjadi lapisan-lapisan di tengah yang disebut **Hidden Layers** (Lapisan Tersembunyi).

Namun, menambahkan *Hidden Layer* menciptakan masalah baru: Bagaimana cara mesin mengoreksi bobot di lapisan pertama, jika kita tidak memiliki data target (kebenaran) untuk lapisan tersembunyi tersebut? Jawabannya adalah algoritma **Backpropagation** (Propagasi Balik).

### 1. Kebutuhan Akan Lapisan Tersembunyi (Hidden Layers)
Dalam arsitektur sederhana (tanpa hidden layer), setiap input berdiri sendiri dan berkontribusi secara independen terhadap output. Mesin tidak bisa memahami "kombinasi" antar-input.
*Hidden layer* bertindak sebagai "kanvas" di mana jaringan saraf menggabungkan berbagai input untuk menciptakan fitur-fitur baru. Misalnya, mesin tidak lagi hanya melihat piksel individual, tetapi lapisan tersembunyi menggabungkan piksel-piksel tersebut untuk mendeteksi "garis melengkung" atau "sudut".

### 2. Kutukan Linearitas dan Fungsi Aktivasi (ReLU)
Ada aturan matematika yang keras: Jika Anda mengalikan masukan dengan matriks bobot, lalu mengalikan hasilnya dengan matriks bobot lain, secara aljabar itu sama saja dengan mengalikannya dengan satu matriks bobot raksasa. Menumpuk banyak lapisan linier tidak akan membuat jaringan lebih pintar, melainkan hanya membuang-buang daya komputasi.
*   **Fungsi Aktivasi (Non-Linearity):** Untuk mematahkan linearitas tersebut, kita menyisipkan sebuah fungsi di antara *hidden layer*. Fungsi yang paling umum digunakan adalah **ReLU (Rectified Linear Unit)**.
*   **Cara Kerja ReLU:** Sangat sederhana. Jika angka yang melewatinya bernilai negatif, ReLU akan mengubahnya menjadi 0. Jika positif, angka tersebut dibiarkan utuh. Dengan mematikan (menjadi 0) sebagian dari jaringan saraf pada kondisi tertentu, jaringan saraf dapat "menekuk" kurva matematika dan memecahkan pola yang sangat kompleks (seperti masalah XOR).

### 3. Mekanisme Algoritma Backpropagation
*Backpropagation* adalah solusi brilian untuk masalah pembagian kesalahan (*Credit Assignment*) di jaringan mendalam.
*   **Error Layer Output:** Jaringan menghitung seberapa salah tebakan akhirnya (`layer_2_error`).
*   **Memutar Balik Arah:** Jaringan mengambil kesalahan akhir tersebut, lalu mengalikannya dengan **matriks bobot yang diputar (Transpose)** untuk mengirimkan kesalahan itu kembali ke *hidden layer*. Mengapa menggunakan matriks bobot yang sama? Karena bobot yang meneruskan sinyal (forward) adalah bobot yang sama yang bertanggung jawab atas kesalahan tersebut, sehingga bobot yang lebih besar harus memikul porsi "kesalahan" yang lebih besar saat kita menghitung mundur (backward).
*   **Turunan Fungsi Aktivasi:** Setelah kesalahan ditarik mundur ke *hidden layer*, kita harus membatalkannya pada node yang "dimatikan" oleh ReLU. Kita menghitung turunan (derivative) dari ReLU, yang intinya berkata: "Jika node ini tadi bernilai 0 saat propagasi maju, maka jangan ubah bobotnya, karena node ini tidak berkontribusi pada kesalahan akhir."

In [ ]:
import numpy as np

# =====================================================================
# REPRODUKSI KODE BAB 6: JARINGAN SARAF MENDALAM & BACKPROPAGATION
# Arsitektur: 3 Lapisan (Layer 0: Input, Layer 1: Hidden, Layer 2: Output)
# Menggunakan Fungsi Aktivasi ReLU
# =====================================================================

np.random.seed(1) # Agar inisialisasi bobot acak selalu sama saat di-run ulang

# Fungsi Aktivasi ReLU: Mengubah angka negatif menjadi 0
def relu(x):
    return (x > 0) * x

# Turunan Fungsi ReLU (Backpropagation): Bernilai 1 jika x > 0, bernilai 0 jika x <= 0
def relu2deriv(output):
    return output > 0

# Dataset Streetlights (Lampu Lalu Lintas)
# 4 baris observasi, masing-masing memiliki 3 input
streetlights = np.array([
    [ 1, 0, 1 ],
    [ 0, 1, 1 ],
    [ 0, 0, 1 ],
    [ 1, 1, 1 ]
])

# Target kebenaran (Apakah aman untuk menyeberang?)
walk_vs_stop = np.array([1, 1, 0, 0]).T

alpha = 0.2
hidden_size = 4 # Kita membuat 4 node di dalam Hidden Layer

# Inisialisasi Matriks Bobot Acak
# Layer 1 Weights: Menghubungkan 3 Input Node ke 4 Hidden Node
weights_0_1 = 2 * np.random.random((3, hidden_size)) - 1
# Layer 2 Weights: Menghubungkan 4 Hidden Node ke 1 Final Output Node
weights_1_2 = 2 * np.random.random((hidden_size, 1)) - 1

print("Memulai Pelatihan Jaringan Saraf Mendalam (Backpropagation)...")
print("-" * 65)

# Looping Pelatihan (Training Loop) selama 60 iterasi
for iteration in range(60):
    layer_2_error = 0

    for i in range(len(streetlights)):
        # =======================================
        # FORWARD PROPAGATION (Propagasi Maju)
        # =======================================
        layer_0 = streetlights[i:i+1] # Mengambil 1 baris observasi

        # Sinyal masuk dari Layer 0 ke Layer 1, lalu dikenakan fungsi non-linear ReLU
        layer_1 = relu(np.dot(layer_0, weights_0_1))

        # Sinyal masuk dari Layer 1 ke Layer 2 (Output Akhir)
        layer_2 = np.dot(layer_1, weights_1_2)

        # =======================================
        # BACKPROPAGATION (Menghitung Error Mundur)
        # =======================================
        # Hitung kesalahan tebakan akhir
        layer_2_delta = layer_2 - walk_vs_stop[i:i+1]
        layer_2_error += np.sum(layer_2_delta ** 2)

        # Tarik mundur kesalahan dari Layer 2 ke Layer 1
        # Mengalikan delta akhir dengan bobot yang ditranspose (.T)
        # Lalu dikalikan dengan turunan ReLU untuk mengabaikan node yang mati
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * relu2deriv(layer_1)

        # =======================================
        # UPDATE BOBOT (Gradient Descent)
        # =======================================
        # Mengubah matriks bobot menggunakan formula: input * delta * alpha
        weights_1_2 -= alpha * layer_1.T.dot(layer_2_delta)
        weights_0_1 -= alpha * layer_0.T.dot(layer_1_delta)

    if iteration % 10 == 9:
        print(f"Iterasi: {iteration+1:<3} | Total Error: {layer_2_error:.4f}")

print("\nPelatihan selesai! Jaringan berhasil memetakan pola yang kompleks.")

### Penjelasan Eksekusi Kode:

1.  **`relu` dan `relu2deriv`:** Kunci dari jaringan saraf yang mendalam. Tanpa `relu`, model tidak akan bisa menyelesaikan dataset ini jika polanya membutuhkan kombinasi silang. `relu2deriv` adalah bagian dari aturan rantai kalkulus (*chain rule*). Fungsi ini menjamin bahwa jika *hidden node* bernilai `0` saat prediksi (dimatikan oleh ReLU), maka node tersebut tidak akan dimarahi atau bobotnya dikoreksi saat backpropagation.
2.  **`layer_1 = relu(np.dot(layer_0, weights_0_1))`:** Matriks `weights_0_1` berukuran 3x4. Ia mengambil 3 angka input dan mentransformasikannya menjadi 4 angka fitur yang baru (di-representasikan sebagai `layer_1`). Jika hasil transformasinya negatif, `relu()` langsung membabatnya menjadi 0.
3.  **`layer_2_delta = layer_2 - walk_vs_stop[i:i+1]`:** Error dihitung persis seperti di Bab 4 dan Bab 5. Ini adalah selisih antara tebakan akhir dengan label kebenaran sesungguhnya.
4.  **`layer_1_delta = layer_2_delta.dot(weights_1_2.T) * relu2deriv(layer_1)`:** Inilah baris kode Backpropagation yang paling magis dan penting. Jaringan bertanya: *"Seberapa besar masing-masing dari 4 hidden node ini berkontribusi terhadap kesalahan di tebakan akhir?"*
    *   Karena `weights_1_2` memetakan 4 hidden node ke 1 output akhir, kita menggunakan metode `.T` (*Transpose*) untuk membalik proses tersebut: memetakan 1 kesalahan kembali menjadi 4 nilai porsi kesalahan (credit assignment).
    *   Sistem secara otomatis mendistribusikan kesalahan secara akurat menelusuri balik jalur koneksi sinyal.
5.  **Pembaruan Gradien Bersamaan:** Di langkah terakhir, baik `weights_1_2` maupun `weights_0_1` diperbarui secara bersamaan berdasarkan `delta` milik mereka masing-masing yang telah dikalkulasi, memastikan seluruh arsitektur jaringan bergerak perlahan menuju error titik 0.